# Config


In [ ]:
%run <EDU_Config>


In [ ]:
# Deployment parameters
# Target lakehouse
target_lakehouse = f"{gold_lakehouse_name}"
target_schema = f"{gold_schema_name}"
logging.info(f"Target lakehouse: {target_lakehouse}, schema: {target_schema or '(none)'}")

# Feature flags
enable_delete_tables = False
enable_create_tables = True


# Drop tables


In [ ]:
if enable_delete_tables:
    tables_to_delete = [
        PUBLICATION_TABLE_NAME,
        "DimSchool",
        "DimStudent",
        "DimTeacher",
        "DimSection",
        "DimContent",
        "DimAssessment",
        "DimWindow",
        "DimDate",
        "BridgeStudentSection",
        "BridgeTeacherSection",
        "FactAttendanceDay",
        "FactAssessment",
        "FactSubmissionState",
        "FactSubmission",
        "FactStudentAction",
        "FactEnrollment",
        "AggSchoolAttendance",
    ]
    for table_name in tables_to_delete:
        full_table_name = get_gold_full_table_name(table_name)
        if spark.catalog.tableExists(full_table_name):
            spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
            logging.info(f"Dropped table: {full_table_name}")
        else:
            logging.info(f"Table {full_table_name} does not exist.")


# Create tables

`EducationGoldModel` declares the thirteen reporting targets that use case 1-6 read, the
two recorded-work facts, the teacher-action fact that use case 7 reads, the enrollment fact that use case 8 reads,
and the publication control table. Existing tables are
appended so that reviewed additive columns are kept; nothing is dropped implicitly.

Every business table carries `snapshot_id` and `as_of`. They identify the refresh that
produced the row; `as_of` is the Gold run cutoff, not an original source observation time.

Additive columns reach an existing table through the append path's `mergeSchema`, which
adds them at the end of the table as nullable. Nothing proves that silently: the
enrichment compares its DataFrame columns against the physical table before every write
and stops if either side is missing a column, so a schema evolution that did not happen
fails the next run instead of producing a table without the new columns.


In [ ]:
class EducationGoldModel(BaseDataModel):
    def get_entities(self) -> List[Tuple[StructType, str, bool]]:
        return [
            (EducationGoldModel.get_dimschool_schema(), "DimSchool", False),
            (EducationGoldModel.get_dimstudent_schema(), "DimStudent", False),
            (EducationGoldModel.get_dimteacher_schema(), "DimTeacher", False),
            (EducationGoldModel.get_dimsection_schema(), "DimSection", False),
            (EducationGoldModel.get_dimcontent_schema(), "DimContent", False),
            (EducationGoldModel.get_dimassessment_schema(), "DimAssessment", False),
            (EducationGoldModel.get_dimwindow_schema(), "DimWindow", False),
            (EducationGoldModel.get_dimdate_schema(), "DimDate", False),
            (EducationGoldModel.get_bridgestudentsection_schema(), "BridgeStudentSection", False),
            (EducationGoldModel.get_bridgeteachersection_schema(), "BridgeTeacherSection", False),
            (EducationGoldModel.get_factattendanceday_schema(), "FactAttendanceDay", False),
            (EducationGoldModel.get_factassessment_schema(), "FactAssessment", False),
            (EducationGoldModel.get_factsubmissionstate_schema(), "FactSubmissionState", False),
            (EducationGoldModel.get_factsubmission_schema(), "FactSubmission", False),
            (EducationGoldModel.get_factstudentaction_schema(), "FactStudentAction", False),
            (EducationGoldModel.get_factenrollment_schema(), "FactEnrollment", False),
            (EducationGoldModel.get_aggschoolattendance_schema(), "AggSchoolAttendance", False),
            (EducationGoldModel.get_goldpublication_schema(), PUBLICATION_TABLE_NAME, False),
        ]

    @staticmethod
    def get_refresh_fields() -> List[StructField]:
        """Refresh identity carried by every business table."""
        return [
            StructField("snapshot_id", StringType(), nullable=False),
            StructField("as_of", TimestampType(), nullable=False),
        ]

    @staticmethod
    def get_dimschool_schema() -> StructType:
        return StructType([
            StructField("school_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("district_id", StringType(), nullable=False),
            StructField("name", StringType(), nullable=False),
            # Postal address for a map. Appended and nullable so an existing table evolves
            # additively; a school without a recorded address keeps its row.
            StructField("street_address", StringType(), nullable=True),
            StructField("city", StringType(), nullable=True),
            StructField("state", StringType(), nullable=True),
            StructField("postal_code", StringType(), nullable=True),
            StructField("county", StringType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimstudent_schema() -> StructType:
        return StructType([
            StructField("student_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("display_name", StringType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimteacher_schema() -> StructType:
        return StructType([
            StructField("teacher_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("name", StringType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimsection_schema() -> StructType:
        return StructType([
            StructField("section_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("school_id", StringType(), nullable=False),
            StructField("course_id", StringType(), nullable=False),
            StructField("subject", StringType(), nullable=False),
            StructField("grade", StringType(), nullable=False),
            StructField("name", StringType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimcontent_schema() -> StructType:
        # description, due_at and max_score support the class review and schedule cases.
        return StructType([
            StructField("content_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("section_id", StringType(), nullable=False),
            StructField("name", StringType(), nullable=False),
            StructField("source_name", StringType(), nullable=False),
            StructField("description", StringType(), nullable=False),
            StructField("due_at", TimestampType(), nullable=True),
            StructField("max_score", DoubleType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimassessment_schema() -> StructType:
        # No instrument version: the current source does not supply one, so no growth
        # comparison may claim that two administrations are equivalent.
        return StructType([
            StructField("assessment_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("subject", StringType(), nullable=False),
            StructField("unit", StringType(), nullable=False),
            StructField("name", StringType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimwindow_schema() -> StructType:
        return StructType([
            StructField("window_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("name", StringType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_dimdate_schema() -> StructType:
        return StructType([
            StructField("date", DateType(), nullable=False, metadata={"primaryKey": True}),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_bridgestudentsection_schema() -> StructType:
        return StructType([
            StructField("enrollment_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("student_id", StringType(), nullable=False),
            StructField("section_id", StringType(), nullable=False),
            StructField("valid_from", DateType(), nullable=False),
            StructField("valid_to", DateType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_bridgeteachersection_schema() -> StructType:
        return StructType([
            StructField("assignment_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("teacher_id", StringType(), nullable=False),
            StructField("section_id", StringType(), nullable=False),
            StructField("valid_from", DateType(), nullable=False),
            StructField("valid_to", DateType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_factattendanceday_schema() -> StructType:
        # One row per eligible learner-school-instructional day. Present, absent and
        # unknown stay separate so a missing observation is never counted as absence.
        return StructType([
            StructField("attendance_day_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("school_id", StringType(), nullable=False),
            StructField("student_id", StringType(), nullable=False),
            StructField("date", DateType(), nullable=False),
            StructField("grade", StringType(), nullable=False),
            StructField("expected_days", LongType(), nullable=False),
            StructField("known_days", LongType(), nullable=False),
            StructField("present_days", LongType(), nullable=False),
            StructField("absent_days", LongType(), nullable=False),
            StructField("unknown_days", LongType(), nullable=False),
            # Absences in the learner's last ATTENDANCE_RULE_WINDOW_DAYS instructional days,
            # ending with this one, so an early-warning rule ("5 or more absences in 20
            # days") is a filter rather than a per-query window calculation. Appended and
            # nullable for the same additive-evolution reason as FactSubmission.
            StructField("rolling_expected_days", LongType(), nullable=True),
            StructField("rolling_absent_days", LongType(), nullable=True),
            StructField("rolling_window_complete", BooleanType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_factassessment_schema() -> StructType:
        # One row per recorded administration. Missing scores stay null, and is_scored
        # names that fact so a measure never has to decide what a null score means.
        return StructType([
            StructField("result_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("school_id", StringType(), nullable=False),
            StructField("student_id", StringType(), nullable=False),
            StructField("date", DateType(), nullable=False),
            StructField("grade", StringType(), nullable=False),
            StructField("assessment_id", StringType(), nullable=False),
            StructField("window_id", StringType(), nullable=False),
            StructField("score", DoubleType(), nullable=True),
            StructField("standard_error", DoubleType(), nullable=True),
            # Whether this administration carries a score. Symmetry with
            # FactSubmission.is_graded, so null handling stays out of every measure.
            StructField("is_scored", BooleanType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_factsubmissionstate_schema() -> StructType:
        # Recorded source state per learner task. Native flags are preserved; they are
        # not converted into an effective missing or late rate.
        return StructType([
            StructField("state_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("obligation_id", StringType(), nullable=False),
            StructField("student_id", StringType(), nullable=False),
            StructField("content_id", StringType(), nullable=False),
            StructField("section_id", StringType(), nullable=False),
            StructField("state", StringType(), nullable=False),
            StructField("source_missing", BooleanType(), nullable=True),
            StructField("source_late", BooleanType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_factsubmission_schema() -> StructType:
        # Attempts stay in their own table so a state row is never multiplied.
        return StructType([
            StructField("submission_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("obligation_id", StringType(), nullable=False),
            StructField("student_id", StringType(), nullable=False),
            StructField("content_id", StringType(), nullable=False),
            StructField("section_id", StringType(), nullable=False),
            StructField("attempt", IntegerType(), nullable=False),
            StructField("submitted_at", TimestampType(), nullable=False),
            StructField("score", DoubleType(), nullable=True),
            StructField("state", StringType(), nullable=False),
            # Coursework achievement, for the student performance review. Appended after
            # the original columns so an existing table evolves additively, and declared
            # nullable because Delta adds a column to a populated table as nullable
            # whatever is declared here. The enrichment asserts the ones that must never
            # be null, so the contract is enforced where it can actually be checked.
            StructField("points_possible", DoubleType(), nullable=True),
            StructField("is_graded", BooleanType(), nullable=True),
            StructField("is_late", BooleanType(), nullable=True),
            StructField("is_latest_attempt", BooleanType(), nullable=True),
            StructField("subject", StringType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_factstudentaction_schema() -> StructType:
        # One row per action a teacher logged for a learner. The before and after windows
        # hold the learner's own attendance in the FOLLOW_UP_WINDOW_DAYS instructional days
        # either side of the action, as additive components, so a follow-up figure is a
        # ratio of sums. They describe what was observed after an action, not its effect.
        return StructType([
            StructField("action_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("student_id", StringType(), nullable=False),
            StructField("school_id", StringType(), nullable=False),
            StructField("section_id", StringType(), nullable=True),
            StructField("teacher_id", StringType(), nullable=False),
            StructField("date", DateType(), nullable=False),
            StructField("grade", StringType(), nullable=False),
            StructField("class_name", StringType(), nullable=True),
            StructField("subject", StringType(), nullable=True),
            StructField("action_type", StringType(), nullable=False),
            StructField("reason", StringType(), nullable=False),
            StructField("reason_category", StringType(), nullable=False),
            StructField("status", StringType(), nullable=False),
            StructField("follow_up_date", DateType(), nullable=True),
            StructField("completed_date", DateType(), nullable=True),
            StructField("outcome_note", StringType(), nullable=True),
            StructField("is_completed", BooleanType(), nullable=False),
            StructField("is_first_in_category", BooleanType(), nullable=False),
            StructField("window_days", LongType(), nullable=False),
            StructField("before_expected_days", LongType(), nullable=False),
            StructField("before_known_days", LongType(), nullable=False),
            StructField("before_absent_days", LongType(), nullable=False),
            StructField("after_expected_days", LongType(), nullable=False),
            StructField("after_known_days", LongType(), nullable=False),
            StructField("after_absent_days", LongType(), nullable=False),
            StructField("is_comparable", BooleanType(), nullable=False),
            # Work in the action's class: the FOLLOW_UP_WORK_PIECES pieces due before it and
            # the same number due after it, and how many of each have no recorded hand-in.
            # Appended and nullable for additive evolution; enrichment asserts them non-null.
            StructField("work_window_pieces", LongType(), nullable=True),
            StructField("before_work_due", LongType(), nullable=True),
            StructField("before_work_missing", LongType(), nullable=True),
            StructField("after_work_due", LongType(), nullable=True),
            StructField("after_work_missing", LongType(), nullable=True),
            StructField("is_work_comparable", BooleanType(), nullable=True),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_factenrollment_schema() -> StructType:
        # One row per school membership per school year it overlaps. entry_type says how the
        # learner arrived compared with the previous school year, exit_type how they left
        # compared with the next. Both are read from membership alone, never predicted.
        return StructType([
            StructField("enrollment_term_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("enrollment_id", StringType(), nullable=False),
            StructField("student_id", StringType(), nullable=False),
            StructField("school_id", StringType(), nullable=False),
            StructField("grade", StringType(), nullable=False),
            StructField("term_id", StringType(), nullable=False),
            StructField("school_year", StringType(), nullable=False),
            StructField("term_start", DateType(), nullable=False),
            StructField("term_end", DateType(), nullable=False),
            StructField("valid_from", DateType(), nullable=False),
            StructField("valid_to", DateType(), nullable=False),
            StructField("entry_type", StringType(), nullable=False),
            StructField("exit_type", StringType(), nullable=False),
            StructField("is_final_grade", BooleanType(), nullable=False),
            StructField("joined_mid_year", BooleanType(), nullable=False),
            StructField("left_mid_year", BooleanType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_aggschoolattendance_schema() -> StructType:
        # School-grade-period aggregate for the district briefing. period_end is exclusive
        # and students is a distinct count, so it must not be summed across rows.
        return StructType([
            StructField("aggregate_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("school_id", StringType(), nullable=False),
            StructField("grade", StringType(), nullable=False),
            StructField("period_start", DateType(), nullable=False),
            StructField("period_end", DateType(), nullable=False),
            StructField("students", LongType(), nullable=False),
            StructField("present_days", LongType(), nullable=False),
            StructField("known_days", LongType(), nullable=False),
            StructField("expected_days", LongType(), nullable=False),
            StructField("unknown_days", LongType(), nullable=False),
        ] + EducationGoldModel.get_refresh_fields())

    @staticmethod
    def get_goldpublication_schema() -> StructType:
        # One row per completed, validated refresh.
        return StructType([
            StructField("snapshot_id", StringType(), nullable=False, metadata={"primaryKey": True}),
            StructField("as_of", TimestampType(), nullable=False),
            StructField("period_start", DateType(), nullable=False),
            StructField("period_end", DateType(), nullable=False),
            StructField("silver_lakehouse_name", StringType(), nullable=False),
            StructField("table_row_counts", StringType(), nullable=False),
            StructField("published_at", TimestampType(), nullable=False),
        ])


In [ ]:
model = EducationGoldModel(spark, target_lakehouse, target_schema)

if enable_create_tables:
    model.create_entities()
    logging.info(f"✅ Created or updated {len(model.get_entities())} Gold tables in {target_lakehouse}.")
else:
    logging.info("ℹ️ Table creation disabled.")


# Validation


In [ ]:
gold_tables = {table_name: get_gold_full_table_name(table_name)
               for _, table_name, _ in model.get_entities()}
display(get_table_counts(gold_tables))
